In [1]:
#Contribution: Multi-Layer Prompt Injection Detection System

#This notebook demonstrates an integrated detection pipeline combining:
#- Rebuff (heuristic detection)
#- PromptInjection (ML-based detection)

#The system evaluates prompts and makes a final decision using ensemble logic.


In [2]:
import os

from rebuff import RebuffSdk
from llm_guard.input_scanners import PromptInjection
from llm_guard.input_scanners.prompt_injection import MatchType

print("Initializing systems...")

# Rebuff (heuristic only — no API needed)
rb = RebuffSdk(
    "",   # no API key
    "",   # no vector DB
    "test-index",
    "gpt-3.5-turbo"
)

# PromptInjection (ML model)
pi_scanner = PromptInjection(threshold=0.5, match_type=MatchType.FULL)

print("Systems initialized ✅")

Initializing systems...
2026-03-24 19:21:06 [debug    ] Initialized classification model device=device(type='mps') model=Model(path='protectai/deberta-v3-base-prompt-injection-v2', subfolder='', revision='89b085cd330414d3e7d9dd787870f315957e1e9f', onnx_path='ProtectAI/deberta-v3-base-prompt-injection-v2', onnx_revision='89b085cd330414d3e7d9dd787870f315957e1e9f', onnx_subfolder='onnx', onnx_filename='model.onnx', kwargs={}, pipeline_kwargs={'batch_size': 1, 'device': device(type='mps'), 'return_token_type_ids': False, 'max_length': 512, 'truncation': True}, tokenizer_kwargs={})
Systems initialized ✅


In [3]:
def analyze_prompt(prompt):
    
    # --- Rebuff (heuristic only) ---
    try:
        rb_result = rb.detect_injection(
            prompt,
            check_vector=False,
            check_llm=False,
            check_heuristic=True
        )
        rb_flag = rb_result.injection_detected
    except Exception as e:
        print("Rebuff error:", e)
        rb_flag = False

    # --- PromptInjection (ML model) ---
    _, is_valid, pi_score = pi_scanner.scan(prompt)
    pi_flag = not is_valid

    # --- Final decision (ensemble logic) ---
    final_flag = rb_flag or pi_flag

    return {
        "prompt": prompt,
        "rebuff_flag": rb_flag,
        "prompt_injection_flag": pi_flag,
        "pi_score": round(pi_score, 2),
        "final_decision": final_flag
    }

In [4]:
def load_prompts(file_path):
    with open(file_path, "r") as f:
        return [line.strip() for line in f.readlines()]

In [5]:
test_prompts = load_prompts("prompts.txt")

print(f"Loaded {len(test_prompts)} prompts")

Loaded 1000 prompts


In [6]:
results = []

for i, p in enumerate(test_prompts):
    result = analyze_prompt(p)
    results.append(result)

    if i % 50 == 0:
        print(f"Processed {i} prompts...")

2026-03-24 19:21:10 [debug    ] No prompt injection detected   highest_score=0.0
Processed 0 prompts...
2026-03-24 19:21:10 [debug    ] No prompt injection detected   highest_score=0.0
2026-03-24 19:21:11 [debug    ] No prompt injection detected   highest_score=0.0
2026-03-24 19:21:12 [debug    ] No prompt injection detected   highest_score=0.0
2026-03-24 19:21:12 [debug    ] No prompt injection detected   highest_score=0.0
2026-03-24 19:21:12 [debug    ] No prompt injection detected   highest_score=0.0
2026-03-24 19:21:13 [debug    ] No prompt injection detected   highest_score=0.0
2026-03-24 19:21:14 [debug    ] No prompt injection detected   highest_score=0.0
2026-03-24 19:21:14 [debug    ] No prompt injection detected   highest_score=0.0
2026-03-24 19:21:15 [debug    ] No prompt injection detected   highest_score=0.0
2026-03-24 19:21:15 [debug    ] No prompt injection detected   highest_score=0.0
2026-03-24 19:21:15 [debug    ] No prompt injection detected   highest_score=0.0
2026-

In [7]:
import pandas as pd

df = pd.DataFrame(results)
df.head()

,prompt,rebuff_flag,prompt_injection_flag,pi_score,final_decision
0,List machine learning if possible.,False,False,0.0,False
1,Compare Python programming with examples.,False,False,0.0,False
2,Outline operating systems without violating ru...,False,False,0.0,False
3,List operating systems in detail.,False,False,0.0,False
4,Analyze operating systems in detail.,False,False,0.0,False


In [8]:
df.to_csv("contribution_results_1000.csv", index=False)

print("Results saved to contribution_results_1000.csv")

print("Total prompts:", len(df))
print("Flagged by Rebuff:", df["rebuff_flag"].sum())
print("Flagged by PromptInjection:", df["prompt_injection_flag"].sum())
print("Final flagged:", df["final_decision"].sum())

Results saved to contribution_results_1000.csv
Total prompts: 1000
Flagged by Rebuff: 9
Flagged by PromptInjection: 64
Final flagged: 64
